In [1]:
# Kiểm tra xem Colab đã nhận GPU T4 chưa
!nvidia-smi

# Cài đặt các thư viện cần thiết (thêm -q để ẩn log cài đặt cho gọn)
!pip install ultralytics mlflow dagshub -q

Tue Aug  4 08:23:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
import os

# 1. Mount toàn bộ Drive
drive.mount('/content/drive')

# 2. Khai báo đường dẫn tới thư mục repo dự án của bạn trên Drive
# VÍ DỤ: Nếu thư mục dự án của bạn tên là "PPE_YOLOv8_Project" nằm ngoài cùng Drive
PROJECT_DIR = '/content/drive/MyDrive/ppe_cv'

# 3. Tạo thư mục nếu nó chưa tồn tại
if not os.path.exists(PROJECT_DIR):
    os.makedirs(PROJECT_DIR)
    print(f"Đã tạo thư mục mới: {PROJECT_DIR}")

# 4. Thay đổi thư mục làm việc hiện tại của Colab vào thư mục project
%cd {PROJECT_DIR}

print(f"✅ Đã trỏ môi trường làm việc vào: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ppe_cv
✅ Đã trỏ môi trường làm việc vào: /content/drive/MyDrive/ppe_cv


In [3]:
import getpass
import os
import mlflow

DAGSHUB_USER = input("Nhập DagsHub Username: ")
DAGSHUB_TOKEN = getpass.getpass("Nhập DagsHub Token: ")
DAGSHUB_REPO = input("Nhập tên DagsHub Repo (VD: ppe-project): ")

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow"

mlflow.set_experiment("YOLOv8m_train_PPE_Baseline")
print("✅ Cấu hình MLflow thành công!")

Nhập DagsHub Username: TQanh23
Nhập DagsHub Token: ··········
Nhập tên DagsHub Repo (VD: ppe-project): ppe_v2
✅ Cấu hình MLflow thành công!


In [4]:
import os
import getpass
import mlflow
import torch
from ultralytics import YOLO
import shutil

In [5]:
torch.cuda.empty_cache()

model = YOLO("yolov8n.pt")

with mlflow.start_run(run_name="yolov8n_baseline_colab"):
    results = model.train(
        data="/content/drive/MyDrive/ppe_cv/ppe_v2/data.yaml", # CHÚ Ý: Đổi thành đường dẫn thật trên Drive của bạn
        epochs=150,
        imgsz=640,
        batch=32,              # Đã tăng lên 16 nhờ VRAM 16GB của Colab
        workers=4,             # Bật lại workers=2 (Colab free thường có 2 nhân CPU)
        cache=False,
        amp=True,
        patience=30,
        project="/content/YOLOv8_Pruning", # Lưu tạm trên môi trường Colab
        name="baseline_run",
        save=True
    )

print("Huấn luyện hoàn tất!")

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/ppe_cv/ppe_v2/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, 

2026/08/04 08:26:26 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/08/04 08:26:26 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(e7dcd02988d24f6c881c43697b0b67ad) to https://dagshub.com/TQanh23/ppe_v2.mlflow
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to /content/YOLOv8_Pruning/baseline_run-2
Starting training for 150 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/150      3.98G      1.832      2.962      1.507        368        640: 100% ━━━━━━━━━━━━ 66/66 14.4it/s 4.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.5it/s 1.6s
                   all        229       2019      0.197      0.298      0.199     0.0982

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/150      4.91G      1.739       1.97      1.426        362        640: 100% ━━━━━━━━━━━━ 66/66 18.7it/s 3.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 

In [6]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [7]:
model_path = "/content/YOLOv8_Pruning/baseline_run-2/weights/best.pt"
model = YOLO(model_path)

In [9]:
%cd /content/drive/MyDrive/ppe_cv/ppe_v2

!python scripts/val.py \
    --weight /content/YOLOv8_Pruning/baseline_run-2/weights/best.pt \
    --data data.yaml \
    --split val \
    --batch 16 \
    --imgsz 640

/content/drive/MyDrive/ppe_cv/ppe_v2
Average Latency per image: 2.05 ms
Approx FPS: 486.75
Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
Model summary (fused): 73 layers, 3,007,403 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 188.0±71.5 MB/s, size: 75.2 KB)
val: Scanning /content/drive/MyDrive/ppe_cv/train_data/valid/labels.cache... 229 images, 19 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 229/229 73.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 13.9it/s 1.1s
                   all        229       2019      0.697      0.657      0.665      0.326
                 boots         93        201      0.571      0.537      0.523       0.21
                gloves         78        157      0.651      0.344       0.41      0.148
                helmet        130        337      0.784      0.801      0.8